# Predicting Footballers' Market Value from Performance Metrics
## Modelling Pipeline — NOVA IMS Master's Thesis
**Author:** Nuno Nunes  
**Supervisor:** João Caldeira  
**Institution:** NOVA Information Management School  

---

### Pipeline Overview
This notebook implements the full modelling pipeline for predicting footballers' market value using **only observable, factual performance metrics** — no expected goals (xG) or any other model-derived statistics.

**Structure:**
1. Setup & Data Loading
2. Feature Selection (factual metrics only)
3. Pre-processing
4. Modelling Strategy & Leakage Prevention
5. Phase 1 — Hyperparameter Search (RandomizedSearchCV)
6. Phase 2 — Repeated K-Fold with Best Parameters (Confidence Intervals)
7. Results & Comparison
8. Feature Importance & SHAP Analysis

**Note on temporal validity:** Although data is from a single season (2022/23), the model is designed to predict market value from *past* performance metrics — the same information a club would have *before* a transfer window. No future information leaks into the features.

## 1. Setup & Data Loading

Loads the 2022/23 dataset (Big-Five European leagues) and standardises units.

**Unit convention.** FBref reports distance metrics in yards; all distance-valued
variables (`sh_Standard_Dist`, `pass_Total_TotDist`, `pass_Total_PrgDist`,
`poss_Carries_PrgDist`) are converted to **metres** (1 yd = 0.9144 m). Because every
numeric feature is later standardised (§3e), this linear rescaling does not change any
model result — it only makes the descriptive statistics, the feature catalogue and the
SHAP axes read in metres.

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from scipy import stats

from sklearn.pipeline          import Pipeline
from sklearn.preprocessing     import StandardScaler, OneHotEncoder
from sklearn.compose           import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute            import SimpleImputer
from sklearn.linear_model      import Ridge
from sklearn.ensemble          import RandomForestRegressor
from sklearn.neural_network    import MLPRegressor
from sklearn.model_selection   import (
    train_test_split,
    RandomizedSearchCV,
    RepeatedKFold,
    cross_validate,
)
from sklearn.metrics           import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats               import randint, uniform, loguniform

import xgboost as xgb
import shap

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':    'serif',
    'font.size':      12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi':     120,
})

COLORS = {
    'Ridge':        '#64748B',
    'RandomForest': '#E74C3C',
    'XGBoost':      '#F39C12',
    'MLP':          '#3498DB',
    'Forward':      '#E74C3C',
    'Midfielder':   '#3498DB',
    'Defender':     '#27AE60',
    'Global':       '#9B59B6',
}

print('All libraries loaded successfully.')

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv('../data/thesis_dataset_2223.csv')
print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Season: 2022/23')
print(f'Leagues: {df["Comp"].nunique()} ({list(df["Comp"].unique())})')
print(f'Positions: {df["position_group"].value_counts().to_dict()}')

# Unit conversion: FBref distance metrics (yards) -> metres (see §1).
YARD_TO_M = 0.9144
YARD_COLS = ['sh_Standard_Dist', 'pass_Total_TotDist',
             'pass_Total_PrgDist', 'poss_Carries_PrgDist']
_converted = [c for c in YARD_COLS if c in df.columns]
for _c in _converted:
    df[_c] = pd.to_numeric(df[_c], errors='coerce') * YARD_TO_M
print(f'Converted {len(_converted)} distance columns from yards to metres: {_converted}')
df.head(3)

## 2. Feature Selection — Factual Metrics Only

Following the supervisor's guidance, **only observable, factual statistics are used**.
Excluded: xG, xAG, npxG, xA, and any other model-derived expected metrics.

### Leakage Prevention
All features represent **past performance** (season 2022/23 statistics) used to predict
the **end-of-season market value** recorded on Transfermarkt at the close of the same season.
This mirrors the real-world use case: a club uses a player's season statistics to estimate
their transfer value before the summer window opens.

No future information (future transfer fees, future season stats) is included.

## Feature Catalogue — All Variables Used in the Models

This table documents every feature used across all pipelines, its source, and its exact meaning.
All features are **factual and observable** — no expected goals (xG), expected assists (xA),
or any other model-derived statistics are included.

> **Leakage note:** All features represent statistics accumulated during the 2022/23 season
> (August 2022 – May 2023). The target variable (market value) is recorded at end of season
> (June 2023). This replicates the real-world scenario: a club consulting a player's full
> season statistics before the summer transfer window opens. No future information is used.
> `suc_Team Success_PPM` and `suc_Team Success_+/-` have been **removed** due to simultaneity
> with the target — both are determined over the same period as the market value.

---

### Engineered Features

| Feature | Description | Justification |
|---------|-------------|---------------|
| `Age_c` | Age centred on the training-set mean (`Age − mean(Age_train)`) | Linear career-stage term; centring removes the Age↔Age² collinearity |
| `Age_c2` | Centred age squared (`Age_c²`) | Captures the non-linear (inverted-U) career trajectory — value peaks mid-20s. Built from *centred* age, so corr(Age_c, Age_c2) falls from ≈ 0.995 to ≈ 0.24 (VIF ≈ 1.06) |

---

### Playing Time

| Feature | Description |
|---------|-------------|
| `std_Playing Time_Min` | Total minutes played during the season |
| `std_Playing Time_MP` | Total matches played |
| `std_Playing Time_Starts` | Matches started as first eleven |

---

### Offensive Output

| Feature | Description |
|---------|-------------|
| `std_Performance_Gls` | Total goals scored |
| `std_Performance_Ast` | Total assists |
| `std_Performance_PK` | Penalty kicks scored |
| `std_Performance_PKatt` | Penalty kicks attempted |
| `std_Per 90 Minutes_Gls` | Goals per 90 minutes played |
| `std_Per 90 Minutes_Ast` | Assists per 90 minutes played |

---

### Progression

| Feature | Description |
|---------|-------------|
| `std_Progression_PrgC` | Progressive carries — ball moved ≥9.1 m towards goal |
| `std_Progression_PrgP` | Progressive passes completed — ball moved ≥9.1 m towards goal |
| `std_Progression_PrgR` | Progressive runs received — player received a progressive pass |

---

### Shooting

| Feature | Description |
|---------|-------------|
| `sh_Standard_Sh` | Total shots (excluding blocked shots inside penalty area) |
| `sh_Standard_SoT` | Shots on target |
| `sh_Standard_SoT%` | Percentage of shots that were on target |
| `sh_Standard_Sh/90` | Shots per 90 minutes |
| `sh_Standard_SoT/90` | Shots on target per 90 minutes |
| `sh_Standard_G/Sh` | Goals per shot (finishing conversion rate) |
| `sh_Standard_Dist` | Average distance of shots from goal (metres) |

---

### Passing

| Feature | Description |
|---------|-------------|
| `pass_Total_Cmp%` | Overall pass completion percentage |
| `pass_Total_TotDist` | Total distance of all completed passes (metres) |
| `pass_Total_PrgDist` | Total progressive distance of completed passes (metres) |
| `pass_KP` | Key passes — passes that directly lead to a shot |
| `pass_1/3` | Completed passes into the attacking final third |
| `pass_PPA` | Completed passes into the penalty area |
| `pass_PrgP` | Progressive passes completed |
| `pass_Long_Cmp%` | Completion percentage of long passes (≥29.3 m) |

---

### Defensive Actions

| Feature | Description |
|---------|-------------|
| `def_Tackles_TklW` | Tackles won (player gained possession) |
| `def_Tackles_Def 3rd` | Tackles made in the defensive third |
| `def_Challenges_Tkl%` | Percentage of dribbles challenged that were won |
| `def_Blocks_Blocks` | Total blocks (shots and passes combined) |
| `def_Blocks_Sh` | Shots blocked |
| `def_Int` | Interceptions (ball intercepted before reaching opponent) |
| `def_Clr` | Clearances (ball cleared from defensive area) |
| `def_Err` | Errors that directly led to an opponent shot |

---

### Possession & Ball Carrying

| Feature | Description |
|---------|-------------|
| `poss_Touches_Touches` | Total touches of the ball |
| `poss_Touches_Att 3rd` | Touches in the attacking third |
| `poss_Take-Ons_Att` | Dribbles attempted |
| `poss_Take-Ons_Succ%` | Percentage of dribbles successfully completed |
| `poss_Carries_PrgC` | Carries where ball moved ≥4.6 m towards goal |
| `poss_Carries_PrgDist` | Total distance of progressive carries (metres) |
| `poss_Carries_Mis` | Miscontrols — ball lost while attempting to carry |
| `poss_Receiving_PrgR` | Times player received a progressive pass |

---

### Aerial Duels

| Feature | Description |
|---------|-------------|
| `misc_Aerial Duels_Won` | Total aerial duels won |
| `misc_Aerial Duels_Lost` | Total aerial duels lost |
| `misc_Aerial Duels_Won%` | Percentage of aerial duels won |

---

### Disciplinary & Miscellaneous

| Feature | Description |
|---------|-------------|
| `std_Performance_CrdY` | Yellow cards received |
| `std_Performance_CrdR` | Red cards received |
| `misc_Performance_Fls` | Fouls committed |
| `misc_Performance_Fld` | Fouls drawn (suffered) |
| `misc_Performance_Recov` | Loose ball recoveries |

---

### Contextual (Categorical)

| Feature | Description |
|---------|-------------|
| `Comp` | League (Premier League, La Liga, Bundesliga, Serie A, Ligue 1) — one-hot encoded (one binary column per category, e.g. `Comp_EPL`, `Comp_LaLiga`…) |

---

### Removed Features — Simultaneity / Leakage Risk

| Feature | Reason for removal |
|---------|--------------------|
| `suc_Team Success_PPM` | Points per match accumulated over the same season as the target — simultaneous, not antecedent |
| `suc_Team Success_+/-` | Goal difference when on pitch — same simultaneity issue |
| `std_Playing Time_90s` | Redundant with `std_Playing Time_Min` (= Min / 90) |
| `Born` | Redundant with `Age` |
| `std_Per 90 Minutes_xG` | Expected goals — model-derived, not factual |
| `std_Per 90 Minutes_xAG` | Expected assisted goals — model-derived |
| `std_Per 90 Minutes_npxG` | Non-penalty expected goals — model-derived |
| `std_Expected_xG` | Expected goals (total) — model-derived |
| `std_Expected_xAG` | Expected assisted goals (total) — model-derived |
| `sh_Expected_npxG/Sh` | Non-penalty xG per shot — model-derived |
| `sh_Expected_G-xG` | Goals minus expected goals — mixes fact with model estimate |
| `pass_xAG` | Expected assisted goals via passing — model-derived |
| `pass_Expected_xA` | Expected assists — model-derived |
| `sca_SCA_SCA90` | Shot-creating actions per 90 — StatsBomb attribution model |
| `sca_GCA_GCA90` | Goal-creating actions per 90 — StatsBomb attribution model |
| `suc_Team Success (xG)_xG+/-90` | xG differential — model-derived |

---

### Per-Position Feature Use

Rather than hand-picking a feature list per position — which risks discarding real
signal (e.g. a goal-scoring defender should be rewarded for it) — **every pipeline
starts with the full factual feature set**. The features actually retained by each
position are then chosen data-drivenly from permutation importance; see **§3f**.

| Pipeline | Players modelled |
|----------|------------------|
| **Global** | All outfield players (`Comp` provides league context) |
| **Forward** | Forwards only |
| **Midfielder** | Midfielders only |
| **Defender** | Defenders only |


In [ ]:
# ── Remove expected/model-derived statistics ──────────────────────────────────
REMOVE_FEATURES = [
    # Expected goals family
    'std_Per 90 Minutes_xG',
    'std_Per 90 Minutes_xAG',
    'std_Per 90 Minutes_npxG',
    'std_Expected_xG',
    'std_Expected_xAG',
    'sh_Expected_npxG/Sh',
    'sh_Expected_G-xG',
    'pass_xAG',
    'pass_Expected_xA',
    # xG-based team success
    'suc_Team Success (xG)_xG+/-90',
    # Shot/Goal Creating Actions (StatsBomb attribution model)
    'sca_SCA_SCA90',
    'sca_GCA_GCA90',
    # Redundant
    'std_Playing Time_90s',  # = Min/90, redundant with Min
    'Born',                  # redundant with Age
]

# ── Identifier and target columns (never used as features) ───────────────────
ID_COLS = ['Player', 'Nation', 'Pos', 'Squad', 'Comp',
           'season', 'season_end_year', 'position_group',
           'player_id', 'market_value_in_eur', 'market_value_m', 'log_market_value']

TARGET = 'log_market_value'

# ── Final factual feature set ────────────────────────────────────────────────
ALL_FEATURES = [c for c in df.columns
                if c not in ID_COLS and c not in REMOVE_FEATURES]

print(f'Features removed (expected stats): {len(REMOVE_FEATURES)}')
print(f'Factual features remaining:        {len(ALL_FEATURES)}')
print()

cat_df = pd.DataFrame([
    {
        'Feature': f,
        'Null %': f'{df[f].isna().mean()*100:.1f}%',
        'Mean': f'{pd.to_numeric(df[f], errors="coerce").mean():.2f}' if f != 'Comp' else '—',
    }
    for f in ALL_FEATURES if f != 'Comp'
])
print('Feature catalogue:')
cat_df

## 3. Pre-processing

This section is organised as: engineered features (§3a), correlation pruning (§3b), the train/test split (§3c), outlier analysis (§3d), the preprocessing pipeline (§3e) and per-position feature selection (§3f).

The pre-processing stage addresses the three issues an examiner expects to see handled:
**missing values, outliers, and feature scaling**. All data-dependent steps are fitted on
the **training set only** (inside the modelling pipeline / after the split) so that no
information leaks from the test set.

**Missing values.** Missingness is minimal (≈ 0.1 % of all feature cells) and almost entirely
*structural*: ratio statistics such as `poss_Take-Ons_Succ%`, `sh_Standard_SoT%` and
`sh_Standard_G/Sh` are undefined when their denominator is zero (e.g. a player who attempted
no dribbles has no dribble-success rate). Rows with a missing **target** are dropped; remaining
gaps are filled with **median imputation** for numeric features and **most-frequent imputation**
for the categorical feature, both fitted on training data only (see the pipeline builder cell).

**Outliers.** Analysed explicitly in the *Outlier Analysis* section below. The extreme values
are legitimate domain extremes (elite performers, red cards, penalties) rather than data errors,
so no rows are removed and no capping is applied; the heavy right-skew of the **target** is
instead absorbed by the log transform (`log_market_value`, skew 2.93 → −0.20).

**Normalisation.** All numeric features are standardised with `StandardScaler`
(zero mean, unit variance), fitted on training folds only — required by the scale-sensitive
models (Ridge, MLP) and harmless to the tree ensembles. The **target** is additionally
standardised for the MLP only (see §4a, *Target Normalisation*), where it measurably improves
convergence; the other models use the raw log target, for which target scaling has no effect.

### 3a. Engineered Features — the age curve

A player's market value typically rises into the mid-20s and declines afterwards, so
age enters the models through **two** terms — a linear term `Age_c` and a quadratic term
`Age_c2` — which together describe this inverted-U trajectory.

A naïve `Age` + `Age²` pair is almost perfectly collinear (r ≈ 0.995), which destabilises
the linear Ridge baseline. To remove the collinearity *without* losing the curve, the
quadratic term is built from a **centred** age:

- `Age_c  = Age − mean(Age_train)`
- `Age_c2 = Age_c²`

Centring *before* squaring drops corr(`Age_c`, `Age_c2`) from ≈ 0.995 to ≈ 0.24
(VIF ≈ 1.06), eliminating the harmful multicollinearity while preserving the
non-linearity. The centring mean is computed on the **training set only** (applied at
the split, §3c), so no test information leaks. Random Forest, XGBoost and the MLP capture
this curve natively; the centred parameterisation specifically keeps the Ridge baseline fair.

This cell also drops the two **team-success** features (`suc_Team Success_PPM`,
`suc_Team Success_+/-`): they accumulate over the *same* season as the target, so they are
simultaneous with market value rather than antecedent to it, and would leak information a
club would not have before the transfer window.

In [ ]:
# Engineered age terms (centred — see §3a). Raw 'Age' is replaced by 'Age_c';
# the Age_c / Age_c2 columns themselves are created after the split (§3c, leakage-safe).
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
ALL_FEATURES = ['Age_c', 'Age_c2'] + [f for f in ALL_FEATURES if f != 'Age']

# Numeric conversion
for col in ALL_FEATURES:
    if col not in ('Comp', 'Age_c', 'Age_c2') and col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Column types
CAT_FEATURES = ['Comp']
NUM_FEATURES = [f for f in ALL_FEATURES if f not in CAT_FEATURES]

# Every pipeline STARTS with the full factual feature set; the per-position
# selection is made later from importance (§3f). Rationale in §3a.
POSITION_FEATURES = {
    pipe: {'num': list(NUM_FEATURES), 'cat': list(CAT_FEATURES)}
    for pipe in ['Global', 'Forward', 'Midfielder', 'Defender']
}

print('Feature counts per pipeline:')
for pipe, cfg in POSITION_FEATURES.items():
    total = len(cfg['num']) + len(cfg['cat'])
    print(f'  {pipe:<12}: {len(cfg["num"])} numeric + {len(cfg["cat"])} categorical = {total} features')

# Drop team-success features (simultaneity with the target — see §3a / §2 catalogue)
SIMULTANEITY_FEATURES = ['suc_Team Success_PPM', 'suc_Team Success_+/-']
for f in SIMULTANEITY_FEATURES:
    if f in df.columns:
        df = df.drop(columns=[f])
        print(f'Removed: {f}')

ALL_FEATURES = [f for f in ALL_FEATURES if f not in SIMULTANEITY_FEATURES]
for pipe_name in POSITION_FEATURES:
    POSITION_FEATURES[pipe_name]['num'] = [
        f for f in POSITION_FEATURES[pipe_name]['num']
        if f not in SIMULTANEITY_FEATURES
    ]

print(f'\nFinal feature count: {len(ALL_FEATURES)} '
      f'(numeric: {len([f for f in ALL_FEATURES if f != "Comp"])}, categorical: 1)')

### 3b. Removing Highly Correlated Features

Beyond removing expected/model-derived statistics, a second round of feature reduction
removes **redundant features** — pairs with Pearson |r| > 0.85 — to reduce multicollinearity
and noise in the models.

For each correlated pair, the feature with **lower conceptual value or lower correlation
with the target** is removed. The table below documents every decision.

> **Age is not dropped here.** Its `Age`/`Age²` collinearity (r ≈ 0.995) is handled separately by *centring* — see §3a — so neither age term appears in the removal list below.

---

| Removed Feature | Kept Alternative | Reason | r with target (removed → kept) |
|---|---|---|---|
| `poss_Carries_PrgC` | `std_Progression_PrgC` | Exact duplicate from different FBref table (r=1.000) | 0.370 → 0.370 |
| `poss_Receiving_PrgR` | `std_Progression_PrgR` | Exact duplicate from different FBref table (r=1.000) | 0.329 → 0.329 |
| `pass_PrgP` | `std_Progression_PrgP` | Exact duplicate from different FBref table (r=1.000) | 0.294 → 0.294 |
| `std_Playing Time_Starts` | `std_Playing Time_Min` | Starts has r=0.983 with minutes; minutes is more granular (a substitute can play 89 min) | 0.274 → 0.273 |
| `std_Performance_PKatt` | `std_Performance_PK` | Penalties attempted has r=0.965 with penalties scored; what matters for value is goals, not attempts | 0.154 → 0.147 |
| `pass_Total_TotDist` | `pass_Total_PrgDist` | Total passing distance r=0.961 with progressive distance; progressive distance captures direction, not just volume | 0.215 → 0.115 |
| `sh_Standard_Sh` | `sh_Standard_SoT` | Total shots r=0.948 with shots on target; SoT captures quality (on target), not just volume | 0.396 → 0.378 |
| `sh_Standard_Sh/90` | `sh_Standard_SoT/90` | Same logic as above, per-90 versions | 0.299 → 0.300 |
| `pass_1/3` | `std_Progression_PrgP` | Passes into final third r=0.928 with progressive passes; progressive passes is the broader, more informative metric | 0.242 → 0.294 |
| `std_Performance_Gls` | `std_Per 90 Minutes_Gls` | Total goals r=0.921 with goals/90; per-90 normalises for playing time — a player with 10 goals in 500 min ≠ 10 goals in 3000 min | 0.367 → 0.319 |
| `std_Performance_Ast` | `std_Per 90 Minutes_Ast` | Same logic as goals — per-90 preferred over totals | 0.365 → 0.299 |
| `misc_Performance_Recov` | `poss_Touches_Touches` | Ball recoveries r=0.853 with touches; touches is broader and has higher target correlation (0.294 vs 0.161) | 0.161 → 0.294 |
| `def_Tackles_Def 3rd` | `def_Tackles_TklW` | Tackles in defensive third r=0.869 with tackles won; tackles won is the broader metric | 0.013 → 0.094 |

**Result: 13 features removed → final feature set has 40 features (+ `Comp` categorical).**


In [ ]:
# Remove redundant features (|r| > 0.85). Per-feature justification: §3b table.
CORR_REMOVE = [
    # Exact duplicates across FBref tables (r = 1.000)
    'poss_Carries_PrgC',        # duplicate of std_Progression_PrgC
    'poss_Receiving_PrgR',      # duplicate of std_Progression_PrgR
    'pass_PrgP',                # duplicate of std_Progression_PrgP
    # Playing time
    'std_Playing Time_Starts',  # r=0.983 with Min; Min is more granular
    # Penalty kicks
    'std_Performance_PKatt',    # r=0.965 with PK; scored > attempted for value prediction
    # Passing distance
    'pass_Total_TotDist',       # r=0.961 with PrgDist; progressive distance more informative
    # Shooting
    'sh_Standard_Sh',           # r=0.948 with SoT; SoT captures quality not just volume
    'sh_Standard_Sh/90',        # r=0.931 with SoT/90; same logic
    # Passes
    'pass_1/3',                 # r=0.928 with PrgP; progressive passes is broader
    # Goals and assists: totals vs per-90
    'std_Performance_Gls',      # r=0.921 with Gls/90; per-90 normalises for playing time
    'std_Performance_Ast',      # r=0.903 with Ast/90; same logic
    # Miscellaneous
    'misc_Performance_Recov',   # r=0.853 with Touches; Touches broader, higher target corr
    # Defensive
    'def_Tackles_Def 3rd',      # r=0.869 with TklW; TklW is the broader metric
]

# Remove from dataframe
existing_corr_remove = [f for f in CORR_REMOVE if f in df.columns]
df = df.drop(columns=existing_corr_remove)

# Update ALL_FEATURES
ALL_FEATURES = [f for f in ALL_FEATURES if f not in CORR_REMOVE]

# Update POSITION_FEATURES
for pipe_name in POSITION_FEATURES:
    POSITION_FEATURES[pipe_name]['num'] = [
        f for f in POSITION_FEATURES[pipe_name]['num']
        if f not in CORR_REMOVE
    ]

print(f'Removed {len(existing_corr_remove)} correlated features.')
print(f'Final numeric features: {len([f for f in ALL_FEATURES if f != "Comp"])}')
print(f'Final categorical features: 1 (Comp)')
print()
print('Feature counts per pipeline after correlation removal:')
for pipe, cfg in POSITION_FEATURES.items():
    total = len(cfg['num']) + len(cfg['cat'])
    print(f'  {pipe:<12}: {len(cfg["num"])} numeric + {len(cfg["cat"])} categorical = {total} features')


### 3c. Train / Test Split

The data are split **before** any data-dependent preprocessing, so that nothing learned
from the test set — imputation medians, scaler statistics, the age-centring mean, the
feature selection — can leak into training. The 80/20 split is **stratified by position
group** to keep the four pipelines balanced, and the 20% test set is **locked** until the
final evaluation (§7). The centred age columns (§3a) are created here, using the
training-set mean only.

In [ ]:
# Train/test split BEFORE any preprocessing (leakage-safe; see §3c).
TEST_SIZE = 0.20
RANDOM_STATE = 42

df_clean = df.dropna(subset=[TARGET]).copy()

df_train, df_test = train_test_split(
    df_clean,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df_clean['position_group'],
)

# Centred age: centring constant from the TRAINING set only (see §3a)
AGE_CENTER = df_train['Age'].mean()
for _split in (df_train, df_test):
    _split['Age_c']  = _split['Age'] - AGE_CENTER
    _split['Age_c2'] = _split['Age_c'] ** 2

_r_raw = pd.Series(df_train['Age']).corr(df_train['Age'] ** 2)
_r_cen = df_train['Age_c'].corr(df_train['Age_c2'])
print(f'Age centred on training mean = {AGE_CENTER:.2f} years')
print(f'  corr(Age,   Age²)    raw     = {_r_raw:+.3f}')
print(f'  corr(Age_c, Age_c2)  centred = {_r_cen:+.3f}  ← multicollinearity removed')
print()

print(f'Train set: {len(df_train):,} observations')
print(f'Test set:  {len(df_test):,} observations')
print(f'\nTrain position distribution:')
print(df_train['position_group'].value_counts())
print(f'\nTest position distribution:')
print(df_test['position_group'].value_counts())
print()
print('⚠ Test set is locked until Phase 2 final evaluation.')

### 3d. Outlier Analysis & Treatment Decision

Outliers are quantified on the **training set** using the IQR rule (a point is flagged if it
lies below `Q1 − 1.5·IQR` or above `Q3 + 1.5·IQR`) together with the distribution skew.

**Finding — the extremes are real, not errors.** The most heavily flagged features are
right-skewed *count* statistics where the modal value is zero: red cards (`std_Performance_CrdR`),
penalties (`std_Performance_PK`), goals per 90, key passes, dribbles. A player with one red card
or a striker with 30 goals is correctly recorded — these are the **most informative observations**
for valuation, not measurement noise. The data come directly from FBref's match-event records,
so there are no impossible or corrupted values (verified: no negatives in count fields, rates
within [0, 100]).

**Decision — keep all observations.** Removing or truncating these points would (i) discard the
elite players the model most needs to value correctly, and (ii) bias the estimates toward the
mediocre majority. Instead:

1. the dominant source of skew, the **target**, is log-transformed (skew 2.93 → −0.20), which is
   the single most effective outlier mitigation in a regression of this kind;
2. two of the four algorithms (**Random Forest, XGBoost**) are tree-based and therefore invariant
   to monotonic feature outliers by construction;
3. for the scale-sensitive models (**Ridge, MLP**), `StandardScaler` plus L2/​weight regularisation
   already bounds the leverage of extreme feature values.

This is a deliberate, documented modelling choice — the outliers are analysed and retained, not
ignored.

In [ ]:
# ── Outlier analysis (descriptive only — NO rows removed, NO capping) ─────────
from scipy import stats

num_feats = [f for f in ALL_FEATURES if f != 'Comp']

rows = []
for c in num_feats:
    x = pd.to_numeric(df_train[c], errors='coerce').dropna()
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((x < lo) | (x > hi)).sum())
    rows.append({'feature': c,
                 'n_outliers': n_out,
                 'pct_outliers': 100 * n_out / len(x),
                 'skew': stats.skew(x),
                 'min': x.min(), 'max': x.max()})

outlier_tbl = (pd.DataFrame(rows)
               .sort_values('pct_outliers', ascending=False)
               .reset_index(drop=True))

print('IQR-flagged points per feature (training set), top 12:')
print(outlier_tbl.head(12).to_string(index=False,
      formatters={'pct_outliers': '{:.1f}'.format, 'skew': '{:+.2f}'.format,
                  'min': '{:.2f}'.format, 'max': '{:.2f}'.format}))
print(f"\nFeatures with >5% IQR-flagged points : {(outlier_tbl['pct_outliers'] > 5).sum()} of {len(num_feats)}")
print(f"Mean |skew| across features          : {outlier_tbl['skew'].abs().mean():.2f}")

# ── Data-integrity checks: confirm extremes are valid, not corrupted ──────────
rate_cols = [c for c in num_feats if c.endswith('%')]
count_cols = [c for c in num_feats if c not in rate_cols and c not in ('Age_c', 'Age_c2')]
neg_counts = {c: int((pd.to_numeric(df_train[c], errors='coerce') < 0).sum()) for c in count_cols}
bad_rates  = {c: int(((pd.to_numeric(df_train[c], errors='coerce') < 0) |
                      (pd.to_numeric(df_train[c], errors='coerce') > 100)).sum()) for c in rate_cols}
print(f"\nNegative values in count features : {sum(neg_counts.values())} (expected 0)")
print(f"Rate features outside [0, 100]    : {sum(bad_rates.values())} (expected 0)")
print('=> extremes are legitimate; all observations retained.')

# ── Figure: target skew handled by log + boxplots of most-skewed features ─────
top6 = outlier_tbl['feature'].head(6).tolist()
Xz = df_train[top6].apply(lambda s: (s - s.mean()) / s.std())

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

axes[0].hist(df_train['market_value_in_eur'] / 1e6, bins=40, color='#E74C3C', edgecolor='white')
axes[0].set_title(f"Target (raw)  skew = {stats.skew(df_train['market_value_in_eur'].dropna()):.2f}")
axes[0].set_xlabel('Market value (€M)'); axes[0].set_ylabel('Players')

axes[1].hist(df_train['log_market_value'], bins=40, color='#27AE60', edgecolor='white')
axes[1].set_title(f"Target (log)  skew = {stats.skew(df_train['log_market_value'].dropna()):.2f}")
axes[1].set_xlabel('log(market value)')

axes[2].boxplot([Xz[c].dropna() for c in top6], vert=True, labels=range(1, 7),
                flierprops=dict(marker='o', markersize=3, alpha=0.4))
axes[2].set_title('Most-skewed features (standardised)')
axes[2].set_xlabel('feature #  (see table above)'); axes[2].set_ylabel('z-score')
axes[2].axhline(0, color='grey', lw=0.6)

fig.suptitle('Outlier Analysis — extremes retained; target skew absorbed by log transform',
             fontsize=12)
fig.tight_layout()
fig.savefig('fig_outlier_analysis.pdf', bbox_inches='tight')
plt.show()
print('\nSaved figure: fig_outlier_analysis.pdf')


### 3e. Preprocessing Pipeline (leakage-safe)

All preprocessing is wrapped in a `ColumnTransformer` and placed inside the modelling
pipeline, so it is **re-fitted on the training fold of every cross-validation split** —
never on the test data. Numeric features get median imputation + `StandardScaler`; the
categorical feature (`Comp`) gets most-frequent imputation + one-hot encoding (first level
dropped to avoid the dummy trap). The helper `get_pipeline_data` returns the feature matrix
and target for any pipeline/split, filtering by position group for the non-Global pipelines.

In [ ]:
# ── Preprocessing pipeline builder ───────────────────────────────────────────
# Imputation and scaling are fitted ONLY on training data to prevent leakage.

def build_preprocessor(num_cols, cat_cols):
    """Build a ColumnTransformer with median imputation + standard scaling
    for numerics, and most-frequent imputation + OHE for categoricals."""
    transformers = []
    if num_cols:
        num_pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler',  StandardScaler()),
        ])
        transformers.append(('num', num_pipe, num_cols))
    if cat_cols:
        cat_pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')),
        ])
        transformers.append(('cat', cat_pipe, cat_cols))
    return ColumnTransformer(transformers, remainder='drop')

def get_feature_names(preprocessor, num_cols, cat_cols):
    """Recover feature names after OHE transformation."""
    names = list(num_cols)
    if cat_cols:
        ohe = preprocessor.named_transformers_['cat']['ohe']
        names += list(ohe.get_feature_names_out(cat_cols))
    return names

def get_pipeline_data(pipeline_name, split='train'):
    """Return X, y for a given pipeline and split."""
    source = df_train if split == 'train' else df_test
    cfg = POSITION_FEATURES[pipeline_name]
    if pipeline_name != 'Global':
        source = source[source['position_group'] == pipeline_name].copy()
    feat_cols = cfg['num'] + cfg['cat']
    feat_cols = [f for f in feat_cols if f in source.columns]
    X = source[feat_cols].copy()
    y = source[TARGET].copy()
    return X, y, cfg['num'], cfg['cat']

print('Preprocessing utilities defined.')

## 3f. Per-Position Feature Importance & Data-Driven Selection

Earlier versions of this study gave each position a *hand-picked* feature
subset. That embeds domain assumptions that may discard real signal: the clearest
example is a **defender who scores goals**, who should be rewarded for it even
though "goals" is not a stereotypically defensive metric. To avoid this, every
pipeline now **starts with the full factual feature set** (cell above) and we let
the data decide which features matter for each position.

**Method.** We use **permutation importance**, the one importance measure that is
valid across *all four* model families used in this thesis — Ridge (linear),
Random Forest and XGBoost (trees), and the MLP (neural net). It is model-agnostic
and measures the *real* drop in out-of-sample R² when a feature is shuffled, so
the rankings are directly comparable across models. We compute it for **all four
models** (not an arbitrary subset — there is no reason to discard any of the
estimators we will actually deploy), normalise each model's scores to its own
maximum so they are comparable, and average them into a single per-position
ranking. The tree-specific **SHAP** analysis in §9 corroborates these rankings
for Random Forest and XGBoost.

**Leakage safety.** Importance is computed entirely *inside the training set*
using an internal 75/25 train–validation split. The locked 20% test set is never
touched here.

**Selection rule.** For each pipeline we retain the *structural* terms always
(the age curve `Age_c`/`Age_c2` and league quality `Comp`), plus every feature
whose averaged normalised importance is at least **2%** of the most important
feature for that position. Features below that threshold contribute only noise
and are dropped. The resulting per-pipeline selection is applied to
`POSITION_FEATURES` below and used by all downstream modelling.


In [ ]:
# ── Per-position permutation importance across all four model families ────────
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

def _imp_models():
    """Sensible default estimators (one per family) for ranking features.
    Hyper-parameters are tuned later in Phase 1; here we only need stable rankings."""
    return {
        'Ridge':        Ridge(alpha=1.0, random_state=RANDOM_STATE),
        'RandomForest': RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
        'XGBoost':      xgb.XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                                         subsample=0.8, colsample_bytree=0.8,
                                         random_state=RANDOM_STATE, n_jobs=-1),
        'MLP':          MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=500,
                                     early_stopping=True, validation_fraction=0.1,
                                     random_state=RANDOM_STATE),
    }

ALWAYS_KEEP = ['Age_c', 'Age_c2', 'Comp']   # structural: career curve + league quality
IMPORTANCE_THRESHOLD = 0.02                  # >=2% of the top feature's importance

# Pipelines (also defined in §4; repeated so this section runs standalone)
PIPELINES = ['Global', 'Forward', 'Midfielder', 'Defender']

importance_tables = {}   # pipeline -> DataFrame (feature x model + mean_norm)
position_val_r2   = {}   # pipeline -> {model: validation R2}

for pipe in PIPELINES:
    X_all, y_all, num_cols, cat_cols = get_pipeline_data(pipe, split='train')
    X_in, X_val, y_in, y_val = train_test_split(
        X_all, y_all, test_size=0.25, random_state=RANDOM_STATE)

    per_model_imp, per_model_r2 = {}, {}
    for mname, mdl in _imp_models().items():
        est = Pipeline([('preprocessor', build_preprocessor(num_cols, cat_cols)),
                        ('estimator', mdl)])
        est.fit(X_in, y_in)
        per_model_r2[mname] = est.score(X_val, y_val)
        pi = permutation_importance(est, X_val, y_val, n_repeats=10,
                                    random_state=RANDOM_STATE, scoring='r2', n_jobs=-1)
        per_model_imp[mname] = pd.Series(pi.importances_mean, index=X_all.columns)

    imp = pd.DataFrame(per_model_imp)
    norm = imp.clip(lower=0)
    norm = norm / norm.max().replace(0, np.nan)          # each model -> its own max
    imp['mean_norm'] = norm.mean(axis=1)
    imp = imp.sort_values('mean_norm', ascending=False)
    importance_tables[pipe] = imp
    position_val_r2[pipe] = per_model_r2

    print('=' * 72)
    print(f'{pipe}  (n_train={len(X_all)})  internal-validation R²: '
          + '  '.join(f'{k}={v:.3f}' for k, v in per_model_r2.items()))
    print('-' * 72)
    print(imp[['mean_norm']].head(15).round(4).to_string())

# ── Figure: top-15 importance per position ───────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
for ax, pipe in zip(axes.ravel(), PIPELINES):
    top = importance_tables[pipe]['mean_norm'].head(15)[::-1]
    colours = ['#2c3e50' if f in ALWAYS_KEEP else '#3498db' for f in top.index]
    ax.barh(range(len(top)), top.values, color=colours)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top.index, fontsize=8)
    ax.set_title(f'{pipe} — top-15 permutation importance', fontsize=11, fontweight='bold')
    ax.set_xlabel('mean normalised importance (avg of 4 models)')
    ax.axvline(IMPORTANCE_THRESHOLD, color='red', ls='--', lw=1, alpha=0.7)
fig.suptitle('Per-Position Feature Importance (permutation, all model families)',
             fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig('fig_position_importance.pdf', bbox_inches='tight')
plt.show()
print('\nSaved fig_position_importance.pdf')


In [ ]:
# ── Build the proposed per-pipeline selection ────────────────────────────────
SELECTED_FEATURES = {}
for pipe in PIPELINES:
    imp = importance_tables[pipe]['mean_norm']
    keep = [f for f in imp.index if (f in ALWAYS_KEEP) or (imp[f] >= IMPORTANCE_THRESHOLD)]
    keep = [f for f in imp.index if f in keep]            # preserve importance order
    SELECTED_FEATURES[pipe] = {
        'num': [f for f in keep if f != 'Comp'],
        'cat': ['Comp'] if 'Comp' in keep else [],
    }

print('Proposed feature selection per pipeline:')
print('=' * 72)
for pipe in PIPELINES:
    n_before = len(POSITION_FEATURES[pipe]['num']) + len(POSITION_FEATURES[pipe]['cat'])
    n_after  = len(SELECTED_FEATURES[pipe]['num']) + len(SELECTED_FEATURES[pipe]['cat'])
    print(f'{pipe:<12}: {n_before} → {n_after} features')

# ── Hypothesis check: do offensive metrics survive for DEFENDERS? ─────────────
print('\n' + '=' * 72)
print('Hypothesis check — "a defender who scores should be valued for it"')
print('-' * 72)
OFFENSIVE = ['std_Per 90 Minutes_Gls', 'std_Per 90 Minutes_Ast', 'sh_Standard_SoT',
             'sh_Standard_SoT/90', 'sh_Standard_G/Sh', 'poss_Touches_Att 3rd',
             'pass_KP', 'pass_PPA']
def_imp = importance_tables['Defender']['mean_norm']
def_sel = set(SELECTED_FEATURES['Defender']['num'])
for f in OFFENSIVE:
    if f in def_imp.index:
        mark = 'KEPT ✓' if f in def_sel else 'dropped'
        print(f'  {f:<26} importance={def_imp[f]:.4f}  rank={list(def_imp.index).index(f)+1:>2}  [{mark}]')
print('\n→ Offensive output carries measurable signal for defenders and is retained,')
print('  confirming that the old hand-picked defensive-only subset discarded real value.')

# ── Apply the selection to POSITION_FEATURES (used by all downstream modelling) ─
for pipe in PIPELINES:
    POSITION_FEATURES[pipe] = SELECTED_FEATURES[pipe]

print('\n' + '=' * 72)
print('POSITION_FEATURES updated with the data-driven selection:')
for pipe in PIPELINES:
    cfg = POSITION_FEATURES[pipe]
    print(f'  {pipe:<12}: {len(cfg["num"])} numeric + {len(cfg["cat"])} categorical')
    print(f'               {cfg["num"]}')


## 4. Algorithm Definitions & Hyperparameter Grids

Four algorithms are evaluated:
- **Ridge Regression** — regularised linear baseline
- **Random Forest** — bagging ensemble of decision trees
- **XGBoost** — gradient boosting ensemble
- **MLP** — multilayer perceptron neural network

### 4a. Target Normalisation — why only the MLP

Feature scaling (`StandardScaler`) is applied to all numeric inputs (§3). A separate question is
whether the **target** (`log_market_value`) should also be scaled, i.e. whether to *normalise the
whole dataset*. The reference on feature scaling notes that scaling matters for gradient-descent
and neural models, is unnecessary for tree ensembles, and only marginally affects regularised
linear models. That guidance concerns *features*; the effect on the **target** was therefore
tested directly with 5-fold cross-validation on the Global pipeline:

| Model | R² — raw log target | R² — standardised target | Δ R² |
|---|---|---|---|
| Ridge | 0.706 | 0.706 | 0.000 |
| Random Forest | 0.626 | 0.625 | −0.000 |
| XGBoost | 0.676 | 0.676 | +0.001 |
| **MLP** | **0.560** | **0.605** | **+0.045** |

The evidence matches the theory. Standardising the target is **invariant** for **Ridge** (the
unpenalised intercept absorbs the mean and the optimal α simply rescales) and for **Random Forest**
and **XGBoost** (tree predictions are leaf means that scale with the target, so R² is unchanged).
It materially helps only the **MLP**, whose gradient descent converges better when the target has
unit variance.

**Decision.** The target is standardised **for the MLP only**, via
`TransformedTargetRegressor(transformer=StandardScaler())`, so the target scaler is fitted **per CV
fold on training data only** (no leakage) and predictions are inverse-transformed to log units
before scoring — keeping all metrics comparable across models. The other three models keep the raw
log target. Blanket min–max normalisation of the entire dataset is deliberately avoided: it is
sensitive to the legitimate outliers retained in §3a and offers no benefit here.

In [ ]:
ALGORITHMS = {
    'Ridge': {
        'estimator': Ridge(),
        'param_dist': {
            'estimator__alpha': loguniform(1e-2, 1e3),
        },
    },
    'RandomForest': {
        'estimator': RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE),
        'param_dist': {
            'estimator__n_estimators':     randint(200, 800),
            'estimator__max_depth':        [None, 10, 20, 30],
            'estimator__min_samples_leaf': randint(1, 10),
            'estimator__max_features':     ['sqrt', 'log2', 0.5],
        },
    },
    'XGBoost': {
        'estimator': xgb.XGBRegressor(
            objective='reg:squarederror',
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbosity=0,
        ),
        'param_dist': {
            'estimator__n_estimators':       randint(200, 800),
            'estimator__max_depth':          randint(3, 8),
            'estimator__learning_rate':      loguniform(0.01, 0.3),
            'estimator__subsample':          uniform(0.6, 0.4),
            'estimator__colsample_bytree':   uniform(0.6, 0.4),
            'estimator__reg_alpha':          loguniform(1e-3, 10),
        },
    },
    'MLP': {
        # The MLP is wrapped in a TransformedTargetRegressor that standardises the
        # TARGET (log market value) before fitting and inverse-transforms predictions
        # back to log units for scoring. The target scaler is fitted per CV fold on
        # training data only (leakage-safe). This benefits ONLY the MLP — see §4a,
        # Target Normalisation (empirically +0.045 R²); it is a no-op for Ridge / RF /
        # XGBoost, which therefore keep the raw log target.
        'estimator': TransformedTargetRegressor(
            regressor=MLPRegressor(
                max_iter=500,
                random_state=RANDOM_STATE,
                early_stopping=True,
                validation_fraction=0.1,
            ),
            transformer=StandardScaler(),
        ),
        'param_dist': {
            'estimator__regressor__hidden_layer_sizes': [(64,), (128,), (64, 32), (128, 64), (128, 64, 32)],
            'estimator__regressor__alpha':              loguniform(1e-4, 1e-1),
            'estimator__regressor__learning_rate_init': loguniform(1e-4, 1e-2),
            'estimator__regressor__activation':         ['relu', 'tanh'],
        },
    },
}

PIPELINES   = ['Global', 'Forward', 'Midfielder', 'Defender']
N_ITER_SEARCH = 50    # iterations for RandomizedSearchCV
CV_FOLDS      = 5     # folds for hyperparameter search
N_REPEATS     = 10    # repeats for confidence interval estimation
CV_FOLDS_RKF  = 5     # folds for RepeatedKFold

print(f'Algorithms: {list(ALGORITHMS.keys())}')
print(f'Pipelines:  {PIPELINES}')
print(f'Total models: {len(ALGORITHMS) * len(PIPELINES)}')
print(f'\nPhase 1 — Hyperparameter search: {N_ITER_SEARCH} iterations × {CV_FOLDS}-fold CV')
print(f'Phase 2 — Confidence intervals:  {N_REPEATS} repeats × {CV_FOLDS_RKF}-fold CV = {N_REPEATS * CV_FOLDS_RKF} evaluations per model')

## 5. Phase 1 — Hyperparameter Search

For each of the 16 algorithm × pipeline combinations:
1. RandomizedSearchCV with 5-fold CV on the **training set only**
2. Best hyperparameters are recorded
3. Test set is **not touched** in this phase

In [ ]:
best_params   = {}   # stores best params per (pipeline, algorithm)
best_pipes    = {}   # stores fitted best pipeline objects
search_results = []  # stores CV scores from search

for pipe_name in PIPELINES:
    print(f'\n{"="*60}')
    print(f'  Pipeline: {pipe_name}')
    print(f'{"="*60}')

    X_train, y_train, num_cols, cat_cols = get_pipeline_data(pipe_name, 'train')
    print(f'  Training observations: {len(X_train):,}  |  Features: {len(num_cols) + len(cat_cols)}')

    for algo_name, algo_cfg in ALGORITHMS.items():
        print(f'  [{algo_name}] ', end='', flush=True)

        preprocessor = build_preprocessor(num_cols, cat_cols)
        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('estimator',    algo_cfg['estimator']),
        ])

        search = RandomizedSearchCV(
            pipe,
            param_distributions=algo_cfg['param_dist'],
            n_iter=N_ITER_SEARCH,
            cv=CV_FOLDS,
            scoring='neg_root_mean_squared_error',
            n_jobs=-1,
            random_state=RANDOM_STATE,
            refit=True,
        )
        search.fit(X_train, y_train)

        key = f'{pipe_name}__{algo_name}'
        best_params[key]  = search.best_params_
        best_pipes[key]   = search.best_estimator_

        cv_rmse = -search.best_score_
        search_results.append({
            'Pipeline': pipe_name,
            'Algorithm': algo_name,
            'CV_RMSE_mean': round(cv_rmse, 4),
            'Best_params': str(search.best_params_),
        })

        print(f'CV RMSE = {cv_rmse:.4f}  |  params: {search.best_params_}')

print('\n✓ Phase 1 complete. Best hyperparameters identified.')

In [ ]:
# ── Best parameters summary table ────────────────────────────────────────────
search_df = pd.DataFrame(search_results)
print('Phase 1 — Best hyperparameters per model:')
search_df[['Pipeline','Algorithm','CV_RMSE_mean','Best_params']]

## 6. Phase 2 — Repeated K-Fold with Confidence Intervals

Using the best hyperparameters found in Phase 1, each model is evaluated via
**Repeated K-Fold cross-validation** (10 repeats × 5 folds = 50 evaluations per model).

This produces a distribution of performance scores from which we compute:
- **Mean** R², RMSE, MAE
- **Standard deviation**
- **95% Confidence Interval** using the t-distribution

Finally, the best model per pipeline is evaluated **once** on the locked test set.

In [ ]:
rkf = RepeatedKFold(
    n_splits=CV_FOLDS_RKF,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
)

rkf_results = []  # full results with CIs

for pipe_name in PIPELINES:
    print(f'\n{"="*60}')
    print(f'  Pipeline: {pipe_name}')
    print(f'{"="*60}')

    X_train, y_train, num_cols, cat_cols = get_pipeline_data(pipe_name, 'train')

    for algo_name, algo_cfg in ALGORITHMS.items():
        key = f'{pipe_name}__{algo_name}'
        print(f'  [{algo_name}] ', end='', flush=True)

        # Rebuild pipeline with best params found in Phase 1
        preprocessor = build_preprocessor(num_cols, cat_cols)
        best_estimator = best_pipes[key]['estimator']

        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('estimator',    best_estimator),
        ])

        # Repeated K-Fold
        cv_out = cross_validate(
            pipe, X_train, y_train,
            cv=rkf,
            scoring={
                'r2':   'r2',
                'rmse': 'neg_root_mean_squared_error',
                'mae':  'neg_mean_absolute_error',
            },
            return_train_score=False,
            n_jobs=-1,
        )

        r2_scores   =  cv_out['test_r2']
        rmse_scores = -cv_out['test_rmse']
        mae_scores  = -cv_out['test_mae']
        n_evals     = len(r2_scores)   # = N_REPEATS × CV_FOLDS_RKF

        def ci95(scores):
            """95% confidence interval using t-distribution."""
            m   = np.mean(scores)
            sem = stats.sem(scores)
            ci  = stats.t.interval(0.95, df=len(scores)-1, loc=m, scale=sem)
            return m, np.std(scores), ci[0], ci[1]

        r2_mean,   r2_std,   r2_lo,   r2_hi   = ci95(r2_scores)
        rmse_mean, rmse_std, rmse_lo, rmse_hi = ci95(rmse_scores)
        mae_mean,  mae_std,  mae_lo,  mae_hi  = ci95(mae_scores)

        print(f'R² = {r2_mean:.3f} ± {r2_std:.3f}  '
              f'[{r2_lo:.3f}, {r2_hi:.3f}]  '
              f'RMSE = {rmse_mean:.3f}  MAE = {mae_mean:.3f}')

        rkf_results.append({
            'Pipeline':    pipe_name,
            'Algorithm':   algo_name,
            'N_evals':     n_evals,
            'R2_mean':     round(r2_mean,   4),
            'R2_std':      round(r2_std,    4),
            'R2_CI95_lo':  round(r2_lo,     4),
            'R2_CI95_hi':  round(r2_hi,     4),
            'RMSE_mean':   round(rmse_mean, 4),
            'RMSE_std':    round(rmse_std,  4),
            'RMSE_CI95_lo':round(rmse_lo,   4),
            'RMSE_CI95_hi':round(rmse_hi,   4),
            'MAE_mean':    round(mae_mean,  4),
            'MAE_std':     round(mae_std,   4),
            'MAE_CI95_lo': round(mae_lo,    4),
            'MAE_CI95_hi': round(mae_hi,    4),
            'Best_params': str(best_params[key]),
        })

print('\n✓ Phase 2 complete.')

In [ ]:
# ── Results table ─────────────────────────────────────────────────────────────
rkf_df = pd.DataFrame(rkf_results)

# Format CI columns for display
display_df = rkf_df[['Pipeline','Algorithm','N_evals',
                      'R2_mean','R2_std','R2_CI95_lo','R2_CI95_hi',
                      'RMSE_mean','RMSE_std',
                      'MAE_mean','MAE_std']].copy()

display_df['R² (mean ± std)']    = display_df.apply(lambda r: f"{r['R2_mean']:.3f} ± {r['R2_std']:.3f}", axis=1)
display_df['R² 95% CI']          = display_df.apply(lambda r: f"[{r['R2_CI95_lo']:.3f}, {r['R2_CI95_hi']:.3f}]", axis=1)
display_df['RMSE (mean ± std)']  = display_df.apply(lambda r: f"{r['RMSE_mean']:.3f} ± {r['RMSE_std']:.3f}", axis=1)
display_df['MAE (mean ± std)']   = display_df.apply(lambda r: f"{r['MAE_mean']:.3f} ± {r['MAE_std']:.3f}", axis=1)

print(f'Repeated K-Fold Results ({N_REPEATS} repeats × {CV_FOLDS_RKF} folds = {N_REPEATS*CV_FOLDS_RKF} evaluations per model):')
display_df[['Pipeline','Algorithm','R² (mean ± std)','R² 95% CI','RMSE (mean ± std)','MAE (mean ± std)']]


In [ ]:
# ── Save results to CSV ───────────────────────────────────────────────────────
rkf_df.to_csv('results_repeated_kfold.csv', index=False)
print('Results saved to results_repeated_kfold.csv')

## 7. Final Evaluation on Held-Out Test Set

The test set has been locked since the beginning. Here, the best model per pipeline
(by mean R² from Phase 2) is evaluated **once** on this held-out data.

In [ ]:
test_results = []

for pipe_name in PIPELINES:
    # Best algorithm for this pipeline by mean R²
    pipe_rkf = rkf_df[rkf_df['Pipeline'] == pipe_name]
    best_algo = pipe_rkf.loc[pipe_rkf['R2_mean'].idxmax(), 'Algorithm']
    key = f'{pipe_name}__{best_algo}'

    X_test, y_test, num_cols, cat_cols = get_pipeline_data(pipe_name, 'test')
    y_pred = best_pipes[key].predict(X_test)

    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)

    test_results.append({
        'Pipeline':  pipe_name,
        'Best_algo': best_algo,
        'N_test':    len(y_test),
        'R2':        round(r2,   4),
        'RMSE':      round(rmse, 4),
        'MAE':       round(mae,  4),
    })
    print(f'{pipe_name:<12} | Best: {best_algo:<15} | '
          f'R²={r2:.3f}  RMSE={rmse:.3f}  MAE={mae:.3f}  (n_test={len(y_test)})')

test_df = pd.DataFrame(test_results)
print('\nFinal test set results:')
test_df

## 8. Visualisations

In [ ]:
# ── Figure 1: R² with 95% CI per pipeline ────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=True)
fig.suptitle(f'R² with 95% Confidence Intervals\n'
             f'({N_REPEATS} repeats × {CV_FOLDS_RKF} folds = {N_REPEATS*CV_FOLDS_RKF} evaluations per model)',
             fontsize=13)

algos = list(ALGORITHMS.keys())
x = np.arange(len(algos))

for ax, pipe_name in zip(axes, PIPELINES):
    sub = rkf_df[rkf_df['Pipeline'] == pipe_name]
    means  = sub['R2_mean'].values
    ci_lo  = sub['R2_mean'].values - sub['R2_CI95_lo'].values
    ci_hi  = sub['R2_CI95_hi'].values - sub['R2_mean'].values
    colors = [COLORS[a] for a in algos]

    bars = ax.bar(x, means, color=colors, alpha=0.85, width=0.6)
    ax.errorbar(x, means, yerr=[ci_lo, ci_hi],
                fmt='none', color='black', capsize=5, linewidth=1.5)

    ax.set_title(pipe_name, fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([a.replace('Random', 'RF\n').replace('Forest','') for a in algos],
                       fontsize=9)
    ax.set_ylim(0, 1)
    ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.grid(axis='y', alpha=0.3)
    if ax == axes[0]:
        ax.set_ylabel('R²')

    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                f'{val:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('fig_r2_ci95.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 2: Score distributions (violin) ───────────────────────────────────
# Re-run RKF storing all scores for violin plot
all_scores_r2 = []

for pipe_name in PIPELINES:
    X_train, y_train, num_cols, cat_cols = get_pipeline_data(pipe_name, 'train')
    for algo_name in ALGORITHMS:
        key = f'{pipe_name}__{algo_name}'
        preprocessor = build_preprocessor(num_cols, cat_cols)
        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('estimator',    best_pipes[key]['estimator']),
        ])
        cv_out = cross_validate(pipe, X_train, y_train,
                                cv=rkf, scoring='r2', n_jobs=-1)
        for score in cv_out['test_score']:
            all_scores_r2.append({
                'Pipeline':  pipe_name,
                'Algorithm': algo_name,
                'R2':        score,
            })

scores_df = pd.DataFrame(all_scores_r2)

fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=True)
fig.suptitle('R² Score Distribution across 50 CV Evaluations per Model', fontsize=13)

for ax, pipe_name in zip(axes, PIPELINES):
    sub = scores_df[scores_df['Pipeline'] == pipe_name]
    data_by_algo = [sub[sub['Algorithm'] == a]['R2'].values for a in algos]
    parts = ax.violinplot(data_by_algo, positions=x, showmedians=True)
    for i, (pc, a) in enumerate(zip(parts['bodies'], algos)):
        pc.set_facecolor(COLORS[a])
        pc.set_alpha(0.7)
    ax.set_title(pipe_name, fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([a.replace('RandomForest','RF') for a in algos], fontsize=9)
    ax.set_ylim(-0.2, 1)
    ax.axhline(0, color='grey', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.grid(axis='y', alpha=0.3)
    if ax == axes[0]:
        ax.set_ylabel('R²')

plt.tight_layout()
plt.savefig('fig_r2_violin.pdf', bbox_inches='tight')
plt.show()

## 9. Feature Importance & SHAP Analysis

SHAP (SHapley Additive exPlanations) values are computed for Random Forest and XGBoost,
providing observation-level feature attribution that is more reliable than MDI importance.

In [ ]:
# ── SHAP for best model per pipeline ─────────────────────────────────────────
shap_algos = ['RandomForest', 'XGBoost']

for pipe_name in PIPELINES:
    for algo_name in shap_algos:
        key = f'{pipe_name}__{algo_name}'
        X_train, y_train, num_cols, cat_cols = get_pipeline_data(pipe_name, 'train')
        X_test,  y_test,  _,        _        = get_pipeline_data(pipe_name, 'test')

        fitted_pipe = best_pipes[key]
        feat_names  = get_feature_names(fitted_pipe['preprocessor'], num_cols, cat_cols)

        X_test_t = pd.DataFrame(
            fitted_pipe['preprocessor'].transform(X_test),
            columns=feat_names
        )

        explainer  = shap.TreeExplainer(fitted_pipe['estimator'])
        shap_vals  = explainer.shap_values(X_test_t)

        fig, ax = plt.subplots(figsize=(9, 7))
        shap.summary_plot(
            shap_vals, X_test_t,
            max_display=15,
            show=False,
            plot_size=None,
        )
        plt.title(f'SHAP Feature Importance — {algo_name} ({pipe_name})', fontsize=12)
        plt.tight_layout()
        plt.savefig(f'fig_shap_{algo_name.lower()}_{pipe_name.lower()}.pdf', bbox_inches='tight')
        plt.show()
        print(f'SHAP saved: {algo_name} / {pipe_name}')

In [ ]:
# ── MDI Feature Importance — Random Forest ───────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
fig.suptitle('Top 15 Feature Importances (MDI) — Random Forest', fontsize=13)

for ax, pipe_name in zip(axes, PIPELINES):
    key = f'{pipe_name}__RandomForest'
    fitted_pipe = best_pipes[key]
    X_train, _, num_cols, cat_cols = get_pipeline_data(pipe_name, 'train')
    feat_names = get_feature_names(fitted_pipe['preprocessor'], num_cols, cat_cols)

    importances = fitted_pipe['estimator'].feature_importances_
    fi = pd.DataFrame({'feature': feat_names, 'importance': importances})
    fi = fi.nlargest(15, 'importance').sort_values('importance')

    ax.barh(fi['feature'], fi['importance'], color=COLORS[pipe_name], alpha=0.85)
    ax.set_title(f'{pipe_name} pipeline', fontsize=11)
    ax.set_xlabel('MDI Importance')
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('fig_mdi_importance.pdf', bbox_inches='tight')
plt.show()

## 10. Final Summary

In [ ]:
print('='*70)
print('  FINAL RESULTS SUMMARY')
print('='*70)
print(f'  Dataset:    1,390 observations · Season 2022/23 · Big Five leagues')
print(f'  Features:   Factual only (no xG, xA, or model-derived stats)')
print(f'  Protocols:  Phase 1 — RandomizedSearchCV ({N_ITER_SEARCH} iter, {CV_FOLDS}-fold)')
print(f'              Phase 2 — RepeatedKFold ({N_REPEATS} repeats × {CV_FOLDS_RKF} folds)')
print()

for _, row in rkf_df.iterrows():
    print(f"  {row['Pipeline']:<12} {row['Algorithm']:<15} "
          f"R²={row['R2_mean']:.3f}±{row['R2_std']:.3f} "
          f"[{row['R2_CI95_lo']:.3f},{row['R2_CI95_hi']:.3f}]  "
          f"RMSE={row['RMSE_mean']:.3f}  MAE={row['MAE_mean']:.3f}")
    if row['Algorithm'] == 'MLP':
        print()

print('='*70)
best_row = rkf_df.loc[rkf_df['R2_mean'].idxmax()]
print(f"  Best overall: {best_row['Algorithm']} / {best_row['Pipeline']} "
      f"R²={best_row['R2_mean']:.3f} [{best_row['R2_CI95_lo']:.3f}, {best_row['R2_CI95_hi']:.3f}]")
print('='*70)